# Исторический код автора

Снимок исходной исследовательской логики. Outputs, attachments и метаданные ядра очищены.
Это не основной маршрут запуска; исходная закрытая Excel-выгрузка не включена в проект.
Исторические обозначения Fine–Gray могут быть методологически неточными: см. docs/methods.md.
Для текущего проекта используйте notebooks/01_prepare_data.ipynb и 02_survival_analysis.ipynb.


In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/project.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри cardio-oncology-survival-analysis")
sys.path.insert(0, str(ROOT / "src"))
from cardio_oncology.paths import project_paths
P = project_paths(ROOT)
P.run.mkdir(parents=True, exist_ok=True)
P.processed.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = P.run
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".matplotlib"))


# Анализ выживаемости: кардиоваскулярные осложнения при РТК

**Входные данные:** `РТК_ССО_подготовленный_v4.csv` (257 пациентов)

**Структура анализа:**
- **Блок 2** — Kaplan–Meier + log-rank: кривые выживаемости по ключевым группам
- **Блок 3** — Cox regression: univariable → multivariable → гендерный субанализ
- **Блок 4** — Субанализ по типам ССО (аритмии, тромбозы, ОКС)
- **Блок 5** — Fine–Gray: конкурирующие риски (смерть от рака до ССО)

**Главный исход:** `event_cardiotox` (кардиотоксичность, выставлена врачом)  
**Время:** `time_days` (дни от начала лечения до первого ССО или конца наблюдения)  
**Конкурирующие риски:** `competing_status` (0=цензура, 1=ССО, 2=смерть без ССО)

## 0. Импорт библиотек и загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
from scipy.stats import mannwhitneyu

from lifelines import KaplanMeierFitter, CoxPHFitter, AalenJohansenFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test

warnings.filterwarnings('ignore')
sns.set(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

# ── Настройки графиков ────────────────────────────────────────────────────────
PALETTE = {'main': '#2C6FAC', 'accent': '#D64E12', 'neutral': '#6C757D',
           'green': '#2E8B57', 'amber': '#D4A017'}
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12,
                     'axes.labelsize': 11, 'font.size': 10})

FILE_PATH = 'РТК_ССО_подготовленный_v4.csv'

In [ ]:
df = pd.read_csv(FILE_PATH)
print(f'Загружено: {df.shape[0]} пациентов × {df.shape[1]} столбцов')
print(f'Кардиотоксичность: {df["event_cardiotox"].sum()} ({df["event_cardiotox"].mean()*100:.1f}%)')
print(f'Медиана наблюдения: {df["time_days"].median():.0f} дней')
print(f'competing_status: {df["competing_status"].value_counts().rename({0:"цензура",1:"ССО",2:"смерть без ССО"}).to_dict()}')

---
# БЛОК 2 — Kaplan–Meier + log-rank

Строим кривые выживаемости (без ССО) для каждой ключевой группы.  
Log-rank тест проверяет: значимо ли различаются кривые между группами.  
**Медиана** = время, к которому 50% пациентов получили ССО.

In [ ]:
# ── Вспомогательная функция: KM-кривые для двух групп ───────────────────────
def km_two_groups(df, col, label0, label1, title, ax,
                  time_col='time_days', event_col='event_cardiotox',
                  color0=None, color1=None):
    """
    Строит KM-кривые для двух групп (col==0 и col==1) на оси ax.
    Возвращает dict с результатами log-rank теста и медианами.
    """
    c0 = color0 or PALETTE['main']
    c1 = color1 or PALETTE['accent']

    sub = df[[time_col, event_col, col]].dropna()
    g0  = sub[sub[col] == 0]
    g1  = sub[sub[col] == 1]

    kmf0 = KaplanMeierFitter()
    kmf1 = KaplanMeierFitter()
    kmf0.fit(g0[time_col], g0[event_col], label=f'{label0} (n={len(g0)})')
    kmf1.fit(g1[time_col], g1[event_col], label=f'{label1} (n={len(g1)})')

    kmf0.plot_survival_function(ax=ax, color=c0, ci_show=True, ci_alpha=0.12)
    kmf1.plot_survival_function(ax=ax, color=c1, ci_show=True, ci_alpha=0.12)

    res = logrank_test(g0[time_col], g1[time_col],
                       g0[event_col], g1[event_col])

    p_str = f'p = {res.p_value:.4f}' if res.p_value >= 0.0001 else 'p < 0.0001'
    ax.set_title(f'{title}\nlog-rank {p_str}', fontsize=11)
    ax.set_xlabel('Дни наблюдения')
    ax.set_ylabel('S(t) — без ССО')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)
    ax.axhline(0.5, color='grey', linestyle='--', alpha=0.5, linewidth=0.8)

    return {
        'group0': label0, 'n0': len(g0),
        'median0': kmf0.median_survival_time_,
        'group1': label1, 'n1': len(g1),
        'median1': kmf1.median_survival_time_,
        'p_value': res.p_value
    }

In [ ]:
# ── 2.0 Общая KM-кривая по всей когорте (без разделения по группам) ──────────
fig, ax = plt.subplots(figsize=(9, 5))

kmf_all = KaplanMeierFitter()
kmf_all.fit(df['time_days'], df['event_cardiotox'],
            label=f'Вся когорта (n={len(df)})')
kmf_all.plot_survival_function(ax=ax, color=PALETTE['main'],
                               ci_show=True, ci_alpha=0.15, linewidth=2)

# Медиана
median_all = kmf_all.median_survival_time_
ax.axhline(0.5, color='grey', linestyle='--', alpha=0.6, linewidth=0.9,
           label='S(t) = 0.50 (медиана)')
ax.axvline(median_all, color=PALETTE['accent'], linestyle='--',
           alpha=0.7, linewidth=0.9,
           label=f'Медиана = {median_all:.0f} дней')

# Доля без ССО на ключевых временны́х точках
for t_ref in [180, 365, 730]:
    s_val = float(kmf_all.survival_function_at_times(t_ref))
    ax.annotate(f'{t_ref} дн.\nS={s_val:.2f}',
                xy=(t_ref, s_val),
                xytext=(t_ref + 60, s_val + 0.05),
                fontsize=8.5, color='#444444',
                arrowprops=dict(arrowstyle='->', color='#999999', lw=0.8))

ax.set_title('Kaplan–Meier: выживаемость без ССО — вся когорта',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Дни наблюдения', fontsize=11)
ax.set_ylabel('S(t) — доля пациентов без ССО', fontsize=11)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)

print(f'Медиана выживаемости без ССО: {median_all:.0f} дней')
print(f'S(t=180):  {float(kmf_all.survival_function_at_times(180)):.3f}')
print(f'S(t=365):  {float(kmf_all.survival_function_at_times(365)):.3f}')
print(f'S(t=730):  {float(kmf_all.survival_function_at_times(730)):.3f}')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'km_overall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.1 Основные KM-кривые: 6 ключевых сравнений ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
km_results = []

km_results.append(km_two_groups(df, 'bevacizumab',
    'Без бевацизумаба', 'Бевацизумаб', 'Бевацизумаб', axes[0,0]))

km_results.append(km_two_groups(df, 'platinum',
    'Без платины', 'Платина', 'Платиносодержащие препараты', axes[0,1]))

km_results.append(km_two_groups(df, 'anemia',
    'Без анемии', 'Анемия', 'Анемия на старте', axes[0,2]))

km_results.append(km_two_groups(df, 'stage_advanced',
    'Стадия I–II', 'Стадия III–IV', 'Стадия заболевания', axes[1,0]))

km_results.append(km_two_groups(df, 'delta_qtc_gt30',
    'ΔQTc < 30 мс', 'ΔQTc ≥ 30 мс', 'Удлинение QTc', axes[1,1]))

km_results.append(km_two_groups(df, 'cvd_history',
    'Без ССЗ в анамнезе', 'ССЗ в анамнезе', 'ССЗ в анамнезе', axes[1,2]))

plt.suptitle('Kaplan–Meier: кривые выживаемости без ССО по ключевым предикторам',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'km_main.png', dpi=150, bbox_inches='tight')
plt.show()

# Сводная таблица
km_df = pd.DataFrame(km_results)
km_df['p_value'] = km_df['p_value'].apply(lambda x: f'{x:.4f}' if x >= 0.0001 else '<0.0001')
print('\nСводная таблица log-rank тестов:')
display(km_df)

In [ ]:
# ── 2.1b Дополнительные KM-кривые: предикторы из списка медиков ──────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# mets_liver — метастазы в печень
km_results_extra = []
km_results_extra.append(km_two_groups(df, 'mets_liver',
    'Без метастазов в печень', 'Метастазы в печень',
    'Метастазы в печень', axes[0,0]))

# delta_qtc_gt60 — опасное удлинение QTc
km_results_extra.append(km_two_groups(df, 'delta_qtc_gt60',
    'ΔQTc < 60 мс', 'ΔQTc ≥ 60 мс',
    'Критическое удлинение QTc (≥60 мс)', axes[0,1]))

# ── Непрерывные предикторы: дихотомизация по клиническим порогам ─────────────
# ЧСС: тахикардия ≥ 100 уд/мин (клинический порог)
df['tachycardia'] = (df['hr'] >= 100).astype('Int64')
km_results_extra.append(km_two_groups(df, 'tachycardia',
    'ЧСС < 100 уд/мин', 'Тахикардия ≥ 100 уд/мин',
    'ЧСС исходная', axes[0,2]))

# Размер ЛП: ≥ 40 мм — клинический порог риска ФП
df['la_enlarged'] = (df['la_size'] >= 40).astype('Int64')
km_results_extra.append(km_two_groups(df, 'la_enlarged',
    'ЛП < 40 мм', 'ЛП ≥ 40 мм (увеличено)',
    'Размер левого предсердия', axes[1,0]))

# QTc исходный: порог 440 мс (мужчины) / 460 мс (женщины)
# Используем уже имеющийся столбец qtc_start_ms, порог 440 как общий
df['qtc_high'] = (df['qtc_start_ms'] >= 440).astype('Int64')
km_results_extra.append(km_two_groups(df, 'qtc_high',
    'QTc < 440 мс', 'QTc ≥ 440 мс',
    'Исходный QTc', axes[1,1]))

# ФВ ЛЖ: 3 группы на одном графике вручную
ax = axes[1,2]
lvef_labels = {0: '>50% норма', 1: '40–50% сниж.', 2: '<40% знач. сниж.'}
lvef_colors = [PALETTE['main'], PALETTE['amber'], PALETTE['accent']]
for lvef_val, (label, color) in zip([0,1,2],
        zip(lvef_labels.values(), lvef_colors)):
    sub = df[df['lvef_cat_num'] == lvef_val]
    if len(sub) < 5: continue
    kmf = KaplanMeierFitter()
    kmf.fit(sub['time_days'], sub['event_cardiotox'],
            label=f'{label} (n={len(sub)})')
    kmf.plot_survival_function(ax=ax, color=color, ci_show=False)
ax.set_title('ФВ ЛЖ (три категории)', fontsize=11)
ax.set_xlabel('Дни наблюдения'); ax.set_ylabel('S(t) — без ССО')
ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
ax.axhline(0.5, color='grey', linestyle='--', alpha=0.5, linewidth=0.8)

plt.suptitle('Kaplan–Meier: дополнительные предикторы из списка медиков',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'km_extra.png', dpi=150, bbox_inches='tight')
plt.show()

# Сводная таблица
km_extra_df = pd.DataFrame(km_results_extra)
km_extra_df['p_value'] = km_extra_df['p_value'].apply(
    lambda x: f'{x:.4f}' if x >= 0.0001 else '<0.0001')
print('\nСводная таблица (дополнительные предикторы):')
display(km_extra_df)

# Очищаем временные столбцы
df.drop(columns=['tachycardia', 'la_enlarged', 'qtc_high'],
        errors='ignore', inplace=True)

In [ ]:
# ── 2.2 KM по схемам химиотерапии (multivariate log-rank) ───────────────────
fig, ax = plt.subplots(figsize=(10, 6))

scheme_order = ['FOLFOX', 'XELOX', 'FOLFIRI', 'DeGramont', 'Другое', 'Нет ХТ']
colors_scheme = ['#1f77b4','#d62728','#2ca02c','#9467bd','#8c564b','#7f7f7f']

for scheme, color in zip(scheme_order, colors_scheme):
    sub = df[df['cht_scheme_label'] == scheme]
    if len(sub) == 0:
        continue
    kmf = KaplanMeierFitter()
    kmf.fit(sub['time_days'], sub['event_cardiotox'],
            label=f'{scheme} (n={len(sub)})')
    kmf.plot_survival_function(ax=ax, color=color, ci_show=False)

# Multivariate log-rank
sub_ml = df[['time_days', 'event_cardiotox', 'cht_scheme_label']].dropna()
res_ml = multivariate_logrank_test(
    sub_ml['time_days'], sub_ml['cht_scheme_label'], sub_ml['event_cardiotox']
)
p_str = f'{res_ml.p_value:.4f}' if res_ml.p_value >= 0.0001 else '<0.0001'

ax.set_title(f'Kaplan–Meier по схемам ХТ\nmultivariate log-rank p = {p_str}',
             fontsize=12)
ax.set_xlabel('Дни наблюдения')
ax.set_ylabel('S(t) — без ССО')
ax.set_ylim(0, 1.05)
ax.axhline(0.5, color='grey', linestyle='--', alpha=0.5, linewidth=0.8)
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'km_schemes.png', dpi=150, bbox_inches='tight')
plt.show()

# Медианы по схемам
med_rows = []
for scheme in scheme_order:
    sub = df[df['cht_scheme_label'] == scheme].dropna(subset=['time_days','event_cardiotox'])
    if len(sub) == 0: continue
    kmf = KaplanMeierFitter()
    kmf.fit(sub['time_days'], sub['event_cardiotox'])
    med_rows.append({'Схема': scheme, 'n': len(sub),
                     'Медиана без ССО (дни)': kmf.median_survival_time_,
                     'События n (%)': f'{sub["event_cardiotox"].sum()} ({sub["event_cardiotox"].mean()*100:.1f}%)'})
print('Медианы выживаемости по схемам ХТ:')
display(pd.DataFrame(med_rows))

In [ ]:
# ── 2.3 KM по полу ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
km_two_groups(df, 'sex', 'Женщины (0)', 'Мужчины (1)',
              'Kaplan–Meier по полу', ax)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'km_sex.png', dpi=150, bbox_inches='tight')
plt.show()

---
# БЛОК 3 — Cox regression

**Шаг 1:** Univariable Cox — каждый предиктор по одному.  
**Шаг 2:** Отбор с p < 0.20 для многофакторной модели.  
**Шаг 3:** Multivariable Cox — очищенные HR (независимые эффекты).  
**Шаг 4:** Гендерный субанализ.

In [ ]:
# ── 3.1 Определение предикторов ──────────────────────────────────────────────
# Числовые предикторы для Cox
# cht_scheme включаем как числовой — в Cox интерпретируется как порядковый тренд.
# Для категориального эффекта используем get_dummies (см. комментарий ниже).
UNIVAR_PREDS = [
    'age', 'sex', 'hr', 'lvef_cat_num', 'la_size',
    'esr_start', 'crp_start', 'wbc_start',
    'bmi', 'hgb_start', 'anemia', 'glucose_start',
    'qtc_start_ms', 'delta_qtc_ms', 'delta_qtc_gt30', 'delta_qtc_gt60',
    'stage_advanced', 'mets_liver',
    'platinum', 'n_courses', 'bevacizumab',
    'cvd_history'
]

# Проверяем наличие столбцов
UNIVAR_PREDS = [c for c in UNIVAR_PREDS if c in df.columns]
print(f'Предикторов для анализа: {len(UNIVAR_PREDS)}')

# Вспомогательная функция для univariable Cox
def univariable_cox(df, predictor,
                    time_col='time_days', event_col='event_cardiotox'):
    sub = df[[time_col, event_col, predictor]].dropna()
    if len(sub) < 20 or sub[event_col].sum() < 5:
        return None
    try:
        cph = CoxPHFitter()
        cph.fit(sub, duration_col=time_col, event_col=event_col)
        s = cph.summary
        row = s.loc[predictor]
        return {
            'Предиктор': predictor,
            'n':         len(sub),
            'Events':    int(sub[event_col].sum()),
            'HR':        round(row['exp(coef)'], 3),
            'CI_lower':  round(row['exp(coef) lower 95%'], 3),
            'CI_upper':  round(row['exp(coef) upper 95%'], 3),
            'p':         round(row['p'], 4)
        }
    except Exception as e:
        return None

In [ ]:
# ── 3.2 Univariable Cox — все предикторы ─────────────────────────────────────
uni_rows = []
for pred in UNIVAR_PREDS:
    result = univariable_cox(df, pred)
    if result:
        uni_rows.append(result)

uni_df = pd.DataFrame(uni_rows).sort_values('p')
uni_df['95% ДИ'] = uni_df.apply(
    lambda r: f'[{r["CI_lower"]:.3f} – {r["CI_upper"]:.3f}]', axis=1
)
uni_df['p_str'] = uni_df['p'].apply(
    lambda x: '<0.0001' if x < 0.0001 else f'{x:.4f}'
)

print('Univariable Cox-регрессия:')
display(uni_df[['Предиктор','n','Events','HR','95% ДИ','p_str']].rename(
    columns={'p_str':'p', 'Events':'Событий'}
))

# Отбор для многофакторной модели
selected = uni_df[uni_df['p'] < 0.20]['Предиктор'].tolist()
print(f'\nОтобрано для multivariable Cox (p < 0.20): {len(selected)}')
print(selected)

In [ ]:
# ── 3.3 Forest plot: univariable результаты ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, max(6, len(uni_df) * 0.45)))

uni_sorted = uni_df.sort_values('HR', ascending=True).reset_index(drop=True)
y_pos = range(len(uni_sorted))

colors = [PALETTE['accent'] if p < 0.05 else
          PALETTE['amber']   if p < 0.20 else
          PALETTE['neutral'] for p in uni_sorted['p']]

ax.barh(y_pos, 0, color='none')
for i, (_, row) in enumerate(uni_sorted.iterrows()):
    ax.plot([row['CI_lower'], row['CI_upper']], [i, i],
            color=colors[i], linewidth=1.5, solid_capstyle='round')
    ax.scatter(row['HR'], i, color=colors[i], s=50, zorder=5)

ax.axvline(1.0, color='black', linewidth=0.8, linestyle='--', alpha=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(uni_sorted['Предиктор'], fontsize=9)
ax.set_xlabel('Hazard Ratio (HR)')
ax.set_title('Forest plot — Univariable Cox\n'
             'Красный: p<0.05 | Жёлтый: p<0.20 | Серый: p≥0.20', fontsize=11)

# HR подписи
for i, (_, row) in enumerate(uni_sorted.iterrows()):
    ax.text(row['CI_upper'] + 0.02, i,
            f'HR={row["HR"]:.2f}', va='center', fontsize=8, color=colors[i])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'forest_univar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3.4 Multivariable Cox ─────────────────────────────────────────────────────
# Используем отобранные предикторы + ключевые клинические (даже если p >= 0.20)
# Клинически важные добавляем вручную: bevacizumab, stage_advanced, age
MUST_INCLUDE = ['bevacizumab', 'stage_advanced', 'age']
MULTIVAR_PREDS = list(dict.fromkeys(selected + MUST_INCLUDE))
MULTIVAR_PREDS = [c for c in MULTIVAR_PREDS if c in df.columns]

df_mv = df[['time_days', 'event_cardiotox'] + MULTIVAR_PREDS].dropna()
print(f'Multivariable Cox: n={len(df_mv)}, событий={df_mv["event_cardiotox"].sum()}')
print(f'Предикторов: {len(MULTIVAR_PREDS)}')

cph_mv = CoxPHFitter(penalizer=0.0)
cph_mv.fit(df_mv, duration_col='time_days', event_col='event_cardiotox')
cph_mv.print_summary()
print(f'\nC-index (concordance): {cph_mv.concordance_index_:.3f}')

In [ ]:
# ── 3.5 Forest plot: multivariable ───────────────────────────────────────────
mv_summary = cph_mv.summary.copy()
mv_summary = mv_summary.sort_values('exp(coef)', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(5, len(mv_summary) * 0.5)))
y_pos = range(len(mv_summary))

colors_mv = [PALETTE['accent'] if p < 0.05 else
             PALETTE['amber']   if p < 0.20 else
             PALETTE['neutral'] for p in mv_summary['p']]

for i, (idx, row) in enumerate(mv_summary.iterrows()):
    ax.plot([row['exp(coef) lower 95%'], row['exp(coef) upper 95%']],
            [i, i], color=colors_mv[i], linewidth=2, solid_capstyle='round')
    ax.scatter(row['exp(coef)'], i, color=colors_mv[i], s=60, zorder=5)
    ax.text(row['exp(coef) upper 95%'] + 0.02, i,
            f"HR={row['exp(coef)']:.2f} [p={row['p']:.3f}]",
            va='center', fontsize=8.5, color=colors_mv[i])

ax.axvline(1.0, color='black', linewidth=1, linestyle='--', alpha=0.6)
ax.set_yticks(y_pos)
ax.set_yticklabels(mv_summary.index, fontsize=10)
ax.set_xlabel('Hazard Ratio (HR) — скорректированный')
ax.set_title('Forest plot — Multivariable Cox (скорректированные HR)\n'
             'Красный: p<0.05 | Жёлтый: p<0.20 | Серый: незначимо', fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'forest_multivar.png', dpi=150, bbox_inches='tight')
plt.show()

# Итоговая таблица multivariable
mv_table = mv_summary[['exp(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']].copy()
mv_table.columns = ['HR', 'CI_lower', 'CI_upper', 'p']
mv_table['95% ДИ'] = mv_table.apply(
    lambda r: f'[{r["CI_lower"]:.3f} – {r["CI_upper"]:.3f}]', axis=1
)
mv_table['p_str'] = mv_table['p'].apply(
    lambda x: '<0.0001' if x < 0.0001 else f'{x:.4f}'
)
print('Multivariable Cox — итоговая таблица:')
display(mv_table[['HR','95% ДИ','p_str']].rename(columns={'p_str':'p'}))

In [ ]:
# ── 3.6 Гендерный субанализ ───────────────────────────────────────────────────
print('=== Multivariable Cox: мужчины ===')
df_m = df[df['sex'] == 1][['time_days', 'event_cardiotox'] + MULTIVAR_PREDS].dropna()
df_m = df_m.drop(columns=['sex'], errors='ignore')
print(f'Мужчины: n={len(df_m)}, событий={df_m["event_cardiotox"].sum()}')
if len(df_m) > 20 and df_m['event_cardiotox'].sum() > 10:
    cph_m = CoxPHFitter(penalizer=0.1)
    cph_m.fit(df_m, duration_col='time_days', event_col='event_cardiotox')
    cph_m.print_summary()

print('\n=== Multivariable Cox: женщины ===')
df_f = df[df['sex'] == 0][['time_days', 'event_cardiotox'] + MULTIVAR_PREDS].dropna()
df_f = df_f.drop(columns=['sex'], errors='ignore')
print(f'Женщины: n={len(df_f)}, событий={df_f["event_cardiotox"].sum()}')
if len(df_f) > 20 and df_f['event_cardiotox'].sum() > 10:
    cph_f = CoxPHFitter(penalizer=0.1)
    cph_f.fit(df_f, duration_col='time_days', event_col='event_cardiotox')
    cph_f.print_summary()

---
# БЛОК 4 — Субанализ по типам ССО

Отдельные Cox-модели для каждого составного исхода.  
Цель: понять, какой *тип* ССО связан с конкретным предиктором.

| Исход | Ожидаемая связь |
|---|---|
| `sso_thrombo_hf` | Бевацизумаб, платина |
| `sso_rhythm_major` | ΔQTc, размер ЛП, ЧСС |
| `sso_acs` | Платина, схема ХТ, ССЗ в анамнезе |

In [ ]:
# ── 4.1 Функция субанализа ────────────────────────────────────────────────────
def subanalysis_cox(df, outcome_col, predictors, title, penalizer=0.1):
    """
    Cox-регрессия для конкретного типа ССО.
    Использует то же time_days — время до первого любого ССО или цензуры.
    Пациенты без данного исхода считаются цензурированными.
    """
    preds = [c for c in predictors if c in df.columns]
    sub   = df[['time_days', outcome_col] + preds].dropna()
    n_ev  = int(sub[outcome_col].sum())

    print(f'\n{"="*60}')
    print(f'{title}')
    print(f'n={len(sub)}, событий={n_ev} ({n_ev/len(sub)*100:.1f}%)')

    if n_ev < 10:
        print(f'ПРОПУСК: слишком мало событий ({n_ev} < 10) для устойчивой модели.')
        return None

    cph = CoxPHFitter(penalizer=penalizer)
    cph.fit(sub, duration_col='time_days', event_col=outcome_col)
    cph.print_summary()
    return cph

In [ ]:
# ── 4.2 Тромбозы + ХСН → роль бевацизумаба и платины ────────────────────────
PREDS_THROMBO = ['bevacizumab', 'platinum', 'n_courses', 'age', 'sex',
                 'stage_advanced', 'cvd_history', 'anemia']

cph_thrombo = subanalysis_cox(
    df, 'sso_thrombo_hf', PREDS_THROMBO,
    'SUB-АНАЛИЗ: Тромбозы + декомпенсация ХСН (sso_thrombo_hf)'
)

In [ ]:
# ── 4.3 Значимые аритмии → роль QTc и размера ЛП ────────────────────────────
PREDS_RHYTHM = ['delta_qtc_gt30', 'delta_qtc_ms', 'qtc_start_ms',
                'la_size', 'hr', 'age', 'sex', 'bevacizumab', 'lvef_cat_num']

cph_rhythm = subanalysis_cox(
    df, 'sso_rhythm_major', PREDS_RHYTHM,
    'SUB-АНАЛИЗ: Значимые аритмии — ФП, ЖТ, ФЖ, AV-блокада (sso_rhythm_major)'
)

In [ ]:
# ── 4.4 ОКС → роль схемы ХТ и ССЗ в анамнезе ────────────────────────────────
PREDS_ACS = ['platinum', 'bevacizumab', 'cht_scheme', 'n_courses',
             'cvd_history', 'age', 'sex', 'stage_advanced']

cph_acs = subanalysis_cox(
    df, 'sso_acs', PREDS_ACS,
    'SUB-АНАЛИЗ: ОКС — ИМспST / ИМбпсST / нестабильная стенокардия (sso_acs)'
)

In [ ]:
# ── 4.5 QTc-субанализ: ΔQTc по схемам ХТ (Mann–Whitney) ─────────────────────
print('QTc-субанализ: ΔQTc по схемам ХТ')
print('='*55)

df_qtc = df.dropna(subset=['delta_qtc_ms', 'cht_scheme_label'])

qtc_rows = []
for scheme in df_qtc['cht_scheme_label'].unique():
    sub = df_qtc[df_qtc['cht_scheme_label'] == scheme]['delta_qtc_ms']
    qtc_rows.append({
        'Схема': scheme, 'n': len(sub),
        'Медиана ΔQTc': round(sub.median(), 1),
        'Q1–Q3': f"{sub.quantile(.25):.1f} – {sub.quantile(.75):.1f}",
        'Max ΔQTc': round(sub.max(), 1)
    })
qtc_tbl = pd.DataFrame(qtc_rows).sort_values('Медиана ΔQTc', ascending=False)
display(qtc_tbl)

# Попарные сравнения через Mann–Whitney
print('\nPairwise Mann–Whitney: FOLFOX vs остальные схемы:')
folfox_delta = df_qtc[df_qtc['cht_scheme_label']=='FOLFOX']['delta_qtc_ms'].dropna()
for scheme in ['XELOX', 'FOLFIRI', 'DeGramont', 'Другое', 'Нет ХТ']:
    other = df_qtc[df_qtc['cht_scheme_label']==scheme]['delta_qtc_ms'].dropna()
    if len(other) < 3: continue
    _, p = mannwhitneyu(folfox_delta, other, alternative='two-sided')
    print(f'  FOLFOX vs {scheme}: p = {p:.4f}' + (' *' if p < 0.05 else ''))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

scheme_order_qtc = ['FOLFOX','XELOX','FOLFIRI','DeGramont','Другое','Нет ХТ']
order_present = [s for s in scheme_order_qtc if s in df_qtc['cht_scheme_label'].unique()]

sns.boxplot(data=df_qtc, x='cht_scheme_label', y='delta_qtc_ms',
            order=order_present, palette='Set2', ax=axes[0])
axes[0].axhline(30, color='orange', linestyle='--', alpha=0.7, label='30 мс')
axes[0].axhline(60, color='red',    linestyle='--', alpha=0.7, label='60 мс')
axes[0].set_title('ΔQTc по схемам ХТ')
axes[0].set_xlabel('Схема ХТ'); axes[0].set_ylabel('ΔQTc (мс)')
axes[0].legend(fontsize=9)

# Доля пациентов с ΔQTc ≥ 30 мс по схемам
df_qtc['qt30'] = (df_qtc['delta_qtc_ms'] >= 30).astype(int)
qt30_by_scheme = df_qtc.groupby('cht_scheme_label')['qt30'].mean() * 100
qt30_by_scheme = qt30_by_scheme.reindex(order_present)
qt30_by_scheme.plot(kind='bar', ax=axes[1], color=PALETTE['accent'], alpha=0.75)
axes[1].set_title('Доля пациентов с ΔQTc ≥ 30 мс (%)')
axes[1].set_xlabel('Схема ХТ'); axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'qtc_subanalysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
# БЛОК 5 — Fine–Gray: конкурирующие риски

Учитываем смерти от рака до ССО (`competing_status = 2`) как конкурирующее событие.  
Сравниваем кумулятивную функцию частоты (CIF) с KM-кривыми.  
Если расхождение мало — Cox давал надёжные оценки.

> **Примечание:** в вашей когорте конкурирующих событий мало (1 пациент).  
> Fine–Gray показан для методологической полноты и воспроизводимости.

In [ ]:
# ── 5.1 Кумулятивные функции частоты (CIF) ───────────────────────────────────
print('competing_status распределение:')
print(df['competing_status'].value_counts().rename(
    {0:'цензура (0)', 1:'ССО (1)', 2:'смерть без ССО (2)'}
))
print()
n_competing = (df['competing_status'] == 2).sum()
print(f'Конкурирующих событий (смерть без ССО): {n_competing}')
if n_competing < 5:
    print('⚠  Мало конкурирующих событий — Fine–Gray даёт нестабильные оценки.')
    print('   Результаты носят иллюстративный характер.')

In [ ]:
# ── 5.2 CIF через Aalen–Johansen estimator ────────────────────────────────────
# lifelines: AalenJohansenFitter строит CIF для конкурирующих рисков
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левая: CIF vs KM
ajf = AalenJohansenFitter(calculate_variance=True)
ajf.fit(df['time_days'], df['competing_status'], event_of_interest=1)

kmf_all = KaplanMeierFitter()
kmf_all.fit(df['time_days'], df['event_cardiotox'])

# KM показывает вероятность события без учёта конкурирующих
# CIF показывает реальную кумулятивную частоту ССО
(1 - kmf_all.survival_function_).plot(
    ax=axes[0], label='KM (1 – S(t))', color=PALETTE['neutral'],
    linestyle='--', linewidth=1.5
)
ajf.plot(
    ax=axes[0], label='Fine–Gray CIF', color=PALETTE['main'], linewidth=2
)
axes[0].set_title('KM vs Fine–Gray CIF\nкумулятивная частота ССО')
axes[0].set_xlabel('Дни'); axes[0].set_ylabel('Вероятность ССО')
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=9)

# Правая: CIF по группам (бевацизумаб)
for bev_val, label, color in [(0,'Без бевацизумаба',PALETTE['main']),
                               (1,'Бевацизумаб',    PALETTE['accent'])]:
    sub = df[df['bevacizumab'] == bev_val]
    if len(sub) == 0: continue
    ajf_sub = AalenJohansenFitter(calculate_variance=False)
    ajf_sub.fit(sub['time_days'], sub['competing_status'], event_of_interest=1)
    ajf_sub.plot(ax=axes[1], label=f'{label} (n={len(sub)})', color=color)

axes[1].set_title('Fine–Gray CIF: бевацизумаб vs без')
axes[1].set_xlabel('Дни'); axes[1].set_ylabel('CIF — вероятность ССО')
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fine_gray_cif.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.3 Сравнение HR (Cox) vs SHR (Fine–Gray): таблица ───────────────────────
# Fine–Gray регрессия реализована через взвешенный Cox в lifelines
# Используем competing_status: создаём индикатор и применяем технику IPCW
# Примечание: при n_competing < 5 оценки нестабильны

print('Сравнение HR из Cox vs SHR из Fine–Gray')
print('='*55)

# Cox HR (уже получены в блоке 3)
cox_hr = mv_table[['HR','CI_lower','CI_upper','p']].copy()

# Fine-Gray через альтернативный подход: Cox с indicator для competing
# Для малого числа competing events (n=1) результат будет близок к Cox
df_fg = df[['time_days', 'competing_status'] + MULTIVAR_PREDS].dropna().copy()

# Кодируем: event=1 если ССО (competing_status=1), иначе 0
df_fg['event_sso'] = (df_fg['competing_status'] == 1).astype(int)

# При competing_status=2 (смерть без ССО): время остаётся, event=0
# Это приближение Fine-Gray без IPCW-весов (валидно при малом числе competing)
cph_fg = CoxPHFitter(penalizer=0.0)
df_fg_fit = df_fg[['time_days','event_sso'] + MULTIVAR_PREDS].copy()
cph_fg.fit(df_fg_fit, duration_col='time_days', event_col='event_sso')

fg_hr = cph_fg.summary[['exp(coef)','exp(coef) lower 95%','exp(coef) upper 95%','p']].copy()
fg_hr.columns = ['SHR','SHR_lo','SHR_hi','p_fg']

# Объединяем
comparison = cox_hr.join(fg_hr, how='inner')
comparison['ΔHR'] = (comparison['SHR'] - comparison['HR']).round(3)
comparison['HR_str']  = comparison.apply(lambda r: f"{r['HR']:.3f} [{r['CI_lower']:.3f}–{r['CI_upper']:.3f}]", axis=1)
comparison['SHR_str'] = comparison.apply(lambda r: f"{r['SHR']:.3f} [{r['SHR_lo']:.3f}–{r['SHR_hi']:.3f}]", axis=1)
comparison['Интерпретация'] = comparison['ΔHR'].abs().apply(
    lambda d: 'Совпадают' if d < 0.05 else ('Умеренное расхождение' if d < 0.15 else 'Значительное расхождение')
)

display(comparison[['HR_str','SHR_str','ΔHR','Интерпретация']].rename(
    columns={'HR_str':'Cox HR [95% ДИ]', 'SHR_str':'Fine-Gray SHR [95% ДИ]', 'ΔHR':'Δ'}
))

print()
print('Вывод: если HR и SHR близки (|Δ| < 0.05) — конкурирующие риски')
print('не искажают результаты Cox-модели, и Cox является достаточным методом.')

---
# Итоговая сводка

In [ ]:
print('=' * 65)
print('ИТОГОВАЯ СВОДКА АНАЛИЗА ВЫЖИВАЕМОСТИ')
print('=' * 65)
print(f'Когорта:               {len(df)} пациентов')
print(f'Кардиотоксичность:     {df["event_cardiotox"].sum()} ({df["event_cardiotox"].mean()*100:.1f}%)')
print(f'Медиана без ССО:       {KaplanMeierFitter().fit(df["time_days"], df["event_cardiotox"]).median_survival_time_:.0f} дней')
print()

print('БЛОК 2 — Kaplan–Meier:')
for _, r in km_df.iterrows():
    sig = ' ← значимо' if float(r['p_value'].replace('<','')) < 0.05 else ''
    print(f'  {r["group0"]} vs {r["group1"]}: p={r["p_value"]}{sig}')

print()
print('БЛОК 3 — Cox multivariable:')
sig_mv = mv_table[mv_table['p'] < 0.05]
if len(sig_mv) > 0:
    for idx, r in sig_mv.iterrows():
        print(f'  {idx}: HR={r["HR"]:.3f} {r["95% ДИ"]}, p={r["p_str"]}')
else:
    print('  Значимых предикторов (p<0.05) не выявлено')
print(f'  C-index модели: {cph_mv.concordance_index_:.3f}')

print()
print('БЛОК 5 — Fine–Gray:')
print(f'  Конкурирующих событий: {n_competing}')
if n_competing < 5:
    print('  Малое число competing events → Cox-оценки считаются надёжными')
print()
print('Сохранённые графики:')
for f in ['km_main.png','km_schemes.png','km_sex.png',
          'forest_univar.png','forest_multivar.png',
          'qtc_subanalysis.png','fine_gray_cif.png']:
    print(f'  {f}')